# Testing AE convnext

In [54]:
import torch
import torch.nn as nn
from models.blocks import ConvNeXtcausal
from torch.nn.utils.parametrizations import weight_norm
from transformers import EncodecModel
import sys
sys.path.append('../stable-audio-3')
from stable_audio_3 import AutoencoderModel
from utils.mel import MelSpectra

In [38]:
mel_extractor = MelSpectra(
    sample_rate=24000,
    n_fft=1024,
    hop_length=256,
    n_mels=128
)

In [ ]:
class EncoderFast(nn.Module):
    def __init__(self, in_channels: int, dim: int, latent_dim: int, inter_channels: int, num_blocks: int):
        super(EncoderFast, self).__init__()

        stride = [4, 4, 4]
        self.conv1 = weight_norm(nn.Conv1d(in_channels, dim, kernel_size=7, padding=6))
        self.blocks = [ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)]
        self.stages= nn.ModuleList()

        for s in stride:
            stage = nn.Sequential(
                *self.blocks,
                nn.Conv1d(dim, dim, kernel_size=s, stride=s, padding=s-1)
            
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.proj = nn.Linear(dim, latent_dim)
    
    def forward(self, x):
        x = self.conv1(x)
        print(x.shape)
        for stage in self.stages:
            x = stage(x)
        print(x.shape)
        #x = x.mean(dim=-1)
        x = x.transpose(1,2)
        print(x.shape)
        x = self.norm(x)
        print(x.shape)
        x = self.proj(x)
        x = x.transpose(1,2)
        return x

In [25]:
x = torch.ones(1, 1, 24000)
model = EncodecModel.from_pretrained('facebook/encodec_24khz')
with torch.no_grad():
    emb = model.encoder(x)
    #emb = (emb - emb.mean(dim=-1, keepdim=True)) / (emb.std(dim=-1, keepdim=True) + 1e-5)
print(f"Encoder output shape: {emb.shape}")
print(f"Encoder output mean: {emb.mean().item():.4f}, std: {emb.std().item():.4f}")

Loading weights: 100%|██████████| 252/252 [00:00<00:00, 3322.05it/s]


Encoder output shape: torch.Size([1, 128, 75])
Encoder output mean: -1.1315, std: 8.3578


In [9]:
sample_rate = 24000
x = torch.randn(1, 1, sample_rate*1)
ae = AutoencoderModel.from_pretrained("same-s")
with torch.no_grad():
    emb = ae.encode(x, sample_rate)
print(f"Autoencoder output shape: {emb.shape}")

/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Autoencoder output shape: torch.Size([1, 256, 12])


In [20]:
class Decoder(nn.Module):
    def __init__(self, in_channels: int, dim: int, shift_dim: int, inter_channels: int, num_blocks: int):
        super(Decoder, self).__init__()
        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv = nn.Conv1d(in_channels, dim, kernel_size=7, padding=0)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.blocks = nn.ModuleList([ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)])
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, shift_dim, bias=False) 
        # (B, shift_dim, T) -> (B, 1 , shift_dim * T)
    
    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv(x)
        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.norm(x)
        x = x.transpose(1, 2)  # (B, dim, T)

        for block in self.blocks:
            x = block(x)

        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.linear1(x)
        x = self.linear2(x) # (B, T, shift_dim)
        x = x.view(x.size(0), -1) # (B, shift_dim * T)

        return x

In [21]:
s_dim = x.size(-1) // emb.size(-1)
decoder = Decoder(in_channels=128, dim=512, shift_dim=s_dim, inter_channels=256, num_blocks=2)
decoder = decoder.to(emb.device) 
y = decoder(emb)
print(f"Decoder output shape: {y.shape}")

Decoder output shape: torch.Size([1, 24000])


In [26]:
with torch.no_grad():
    y = model.decoder(emb)

In [27]:
import torch.nn.functional as F
mel_original = mel_extractor(x)
mel_reconstructed = mel_extractor(y)
print(f"EnCodec native mel loss: {F.l1_loss(mel_reconstructed, mel_original).item():.4f}")

EnCodec native mel loss: 53.4766


In [60]:
from encodec import EncodecModel
from encodec.utils import convert_audio
import torchaudio

x, sr = torchaudio.load("/home/lois/wavenext/logs/28-05_at_01_25_12/wavenext/version_0/audio_epoch_30/sample_0_real.wav")
wav = convert_audio(x, sr, model.sample_rate, model.channels)
wav = wav.unsqueeze(0)

model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav)  # liste de (codes, scale)
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    
    # decode : [n_q, B, T] → [B, 128, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))
    print(f"emb shape : {emb.shape}")  # [1, 128, 75]

emb shape : torch.Size([1, 128, 75])
